# 04b — WITH imbalance handling: Swin-Base (×3)

'With handling' arm for swin_base. Loss = class-weighted CrossEntropyLoss **plus label smoothing 0.1** (label smoothing kept identical to the 'without' arm so the ONLY difference between arms is the class weighting — a clean controlled comparison). AdamW 1e-5, cosine, num_workers=0. Files saved with `_bal` suffix.

In [1]:
# CONFIG
DATA_ROOT = "/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset"
SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"
OUT_DIR   = "/kaggle/working"
SEEDS     = [42, 123, 2025]
EPOCHS    = 10
BATCH     = 16
NUM_CLASSES = 5
NUM_WORKERS = 0

import os
for pth in [DATA_ROOT, SPLIT_DIR]:
    assert os.path.isdir(pth), f"Missing path: {pth}"
print("Paths OK | seeds:", SEEDS, "| workers:", NUM_WORKERS)

Paths OK | seeds: [42, 123, 2025] | workers: 0


In [2]:
!pip install timm --quiet
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim, timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score,
    precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_idx = np.load(f"{SPLIT_DIR}/clean_train_indices.npy").tolist()
test_idx  = np.load(f"{SPLIT_DIR}/clean_test_indices.npy").tolist()
class_names = open(f"{SPLIT_DIR}/class_names.txt").read().splitlines()
SEVERE_IDX = class_names.index("Severe")
print("Classes:", class_names)

# --- inverse-frequency class weights from the TRAINING split (single-layer handling) ---
base = datasets.ImageFolder(DATA_ROOT)
all_labels = [base.samples[i][1] for i in range(len(base.samples))]
train_labels = [all_labels[i] for i in train_idx]
counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(float)
inv = 1.0 / counts
class_weights = torch.tensor(inv / inv.sum(), dtype=torch.float).to(device)
print("Class weights (inverse freq, normalised):",
      {class_names[i]: round(float(class_weights[i]),4) for i in range(NUM_CLASSES)})

Device: cuda
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Class weights (inverse freq, normalised): {'Mild': 0.2334, 'Moderate': 0.312, 'No_DR': 0.1272, 'Proliferate_DR': 0.1132, 'Severe': 0.2142}


In [3]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    g = torch.Generator(); g.manual_seed(seed)
    return g

In [4]:
NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
def make_transforms(size, crop_from=None):
    cf = crop_from or int(size * 1.15)
    train_tf = transforms.Compose([
        transforms.Resize((cf, cf)), transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20), transforms.ColorJitter(0.3,0.3,0.2,0.05),
        transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
        transforms.GaussianBlur(3, sigma=(0.1,1.0)),
        transforms.ToTensor(), NORM])
    test_tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), NORM])
    return train_tf, test_tf

def make_loaders(size, seed, crop_from=None):
    g = set_seed(seed)
    train_tf, test_tf = make_transforms(size, crop_from)
    train_ds = Subset(datasets.ImageFolder(DATA_ROOT, transform=train_tf), train_idx)
    test_ds  = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  test_idx)
    return (DataLoader(train_ds, BATCH, shuffle=True, num_workers=NUM_WORKERS, generator=g),
            DataLoader(test_ds,  BATCH, shuffle=False, num_workers=NUM_WORKERS))

In [5]:
def train_model(model, loader, criterion, optimizer, scheduler=None, aux=False, epochs=EPOCHS):
    model.train()
    for ep in range(epochs):
        run = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            if aux and isinstance(out, tuple):
                loss = criterion(out[0], y) + 0.4 * criterion(out[1], y)
            else:
                loss = criterion(out.logits if hasattr(out,"logits") else out, y)
            loss.backward(); optimizer.step()
            run += loss.item()
        if scheduler: scheduler.step()
        print(f"    epoch {ep+1}/{epochs} loss {run/len(loader):.4f}")
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        out = model(x.to(device))
        out = out.logits if hasattr(out,"logits") else out
        P.extend(torch.softmax(out,1).cpu().numpy()); Y.extend(y.numpy())
    return np.array(P), np.array(Y)

def save_and_report(name, seed, model, probs, labels):
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro")
    pr,rc,f1,sup = precision_recall_fscore_support(labels,preds,labels=range(NUM_CLASSES),zero_division=0)
    torch.save(model.state_dict(), f"{OUT_DIR}/{name}_bal_seed{seed}.pth")
    np.savez(f"{OUT_DIR}/{name}_bal_seed{seed}_preds.npz", probs=probs, preds=preds, labels=labels)
    print(f"  [{name}+bal seed {seed}] acc {acc:.4f} | macroF1 {f1m:.4f} | "
          f"SevereRec {rc[SEVERE_IDX]:.3f} | ProlifRec {rc[class_names.index('Proliferate_DR')]:.3f}")
    print(f"  saved {name}_bal_seed{seed}.pth + _preds.npz")

def already_done(name, seed):
    p = f"{OUT_DIR}/{name}_bal_seed{seed}_preds.npz"
    if os.path.exists(p):
        print(f"  [skip] {name}+bal seed {seed} already done"); return True
    return False

### swin_base × 3 (weighted loss + label smoothing)

In [6]:
for s in SEEDS:
    if already_done("swin_base", s): continue
    print(f"swin_base+bal seed {s}"); tr, te = make_loaders(224, s)
    m = timm.create_model("swin_base_patch4_window7_224", pretrained=True, num_classes=NUM_CLASSES).to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    opt = optim.AdamW(m.parameters(), lr=1e-5, weight_decay=0.05)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    m = train_model(m, tr, crit, opt, sch)
    probs, labels = evaluate(m, te); save_and_report("swin_base", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

swin_base+bal seed 42


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

    epoch 1/10 loss 0.7104
    epoch 2/10 loss 0.5175
    epoch 3/10 loss 0.4991
    epoch 4/10 loss 0.4860
    epoch 5/10 loss 0.4857
    epoch 6/10 loss 0.4711
    epoch 7/10 loss 0.4710
    epoch 8/10 loss 0.4667
    epoch 9/10 loss 0.4648
    epoch 10/10 loss 0.4640
  [swin_base+bal seed 42] acc 0.9552 | macroF1 0.9454 | SevereRec 0.793 | ProlifRec 0.976
  saved swin_base_bal_seed42.pth + _preds.npz
swin_base+bal seed 123
    epoch 1/10 loss 0.7685
    epoch 2/10 loss 0.5129
    epoch 3/10 loss 0.4949
    epoch 4/10 loss 0.4847
    epoch 5/10 loss 0.4760
    epoch 6/10 loss 0.4736
    epoch 7/10 loss 0.4694
    epoch 8/10 loss 0.4673
    epoch 9/10 loss 0.4619
    epoch 10/10 loss 0.4636
  [swin_base+bal seed 123] acc 0.9608 | macroF1 0.9485 | SevereRec 0.805 | ProlifRec 0.982
  saved swin_base_bal_seed123.pth + _preds.npz
swin_base+bal seed 2025


    epoch 1/10 loss 0.7480
    epoch 2/10 loss 0.5171
    epoch 3/10 loss 0.4921
    epoch 4/10 loss 0.4817
    epoch 5/10 loss 0.4748
    epoch 6/10 loss 0.4729
    epoch 7/10 loss 0.4655
    epoch 8/10 loss 0.4683
    epoch 9/10 loss 0.4645
    epoch 10/10 loss 0.4660
  [swin_base+bal seed 2025] acc 0.9590 | macroF1 0.9568 | SevereRec 0.920 | ProlifRec 0.915
  saved swin_base_bal_seed2025.pth + _preds.npz


In [7]:
print("swin_base+bal done. Save Version (Commit).")

swin_base+bal done. Save Version (Commit).
